# Website Selection

In this project, I will analyze publicly accessible Personal Services websites.
The selected websites represent different types of personal services.

Only authorized/public homepages will be used for the audit.

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd


In [ ]:
import os
from google import genai

os.environ["GEMINI_API_KEY"] = "MY_API KEY"

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

print("Gemini API connected successfully!")

Gemini API connected successfully!


In [3]:
# Websites URLS we are going to use for scraping data
websites = [
    "https://karachitutors.com/",
    "https://dossaniplus.com/",
    "https://fitathome.pk/",
    "https://safaiwala.pk/",
    "https://timesconsultant.com/"
]


In [4]:
# Verifying the URls  if they are properly store or not
for i, website in enumerate(websites, start=1):
    print(i, website)

1 https://karachitutors.com/
2 https://dossaniplus.com/
3 https://fitathome.pk/
4 https://safaiwala.pk/
5 https://timesconsultant.com/


# Website Quality Criteria

To evaluate Personal Services websites, I will define clear criteria
that can help identify whether a website appears professional, basic,
or outdated.

The audit will focus on six main criteria:
1. Website Completeness
2. Contact and Business Information
3. Call-to-Action (CTA)
4. Social Media Presence
5. Mobile Responsiveness
6. Visual and Content Quality

## Evaluation Criteria Summary

| Criteria | Professional | Basic | Outdated |
|---|---|---|---|
| Website Completeness | Important pages and information are available | Some pages or information are missing | Many important pages are missing or links may be broken |
| Contact Information | Phone, email, location, or contact form is clearly available | Limited contact information | Contact information is missing or difficult to find |
| CTA | Clear actions such as Book Now or Contact Us | Limited CTA options | No clear CTA |
| Social Media | Relevant social links are available | Limited social links | No visible or outdated social links |
| Mobile Responsiveness | Mobile-friendly indicators and adaptable layout | Some mobile-friendly indicators | Few or no mobile-friendly indicators |
| Visual & Content Quality | Clear, organized, readable, and consistent | Simple or limited presentation | Poorly organized, outdated, or unclear presentation |

In [ ]:
#Fetch Website HTML and Extract Factual Signals
url = websites[0]

response = requests.get(url, timeout=10)

print("Website:", url)
print("Status Code:", response.sta200tus_code)

Website: https://karachitutors.com/
Status Code: 200


In [6]:
#Check HTML
html = response.text

print("HTML downloaded successfully.")
print("HTML length:", len(html))

HTML downloaded successfully.
HTML length: 178557


In [7]:
#Parse HTML with BeautifulSoup
soup = BeautifulSoup(html, "html.parser")

print("HTML parsed successfully.")

HTML parsed successfully.


In [8]:
#Find all links in the HTML
links = soup.find_all("a")

print("Total links found:", len(links))

Total links found: 29


In [9]:
# Extract links url
link_urls = []

for link in links:
    href = link.get("href")
    
    if href:
        link_urls.append(href)

print("Links with URLs:", len(link_urls))

Links with URLs: 29


In [10]:
#Checking the links
for link in link_urls[:29]:
    print(link)

#content
mailto:karachitutors.com@gmail.com
https://karachitutors.com/
http://karachitutors.com/
https://karachitutors.com/teacher-registration
https://karachitutors.com/student-registration
https://karachitutors.com/rules-for-teachers
https://karachitutors.com/feed-back
https://karachitutors.com/faqs
mailto:karachitutors.com@gmail.com
https://karachitutors.com/
https://karachitutors.com/student-registration
https://karachitutors.com/contact-us
https://karachitutors.com/rules-for-teachers-2
https://karachitutors.com/faqs
https://karachitutors.com/feed-back
https://karachitutors.com/
https://karachitutors.com/rules-for-parents
https://karachitutors.com/rules-for-teachers
https://karachitutors.com/teacher-registration
https://karachitutors.com/tutor-academy-in-karachi
https://karachitutors.com/teacher-registration.php
https://karachitutors.com/student-registration.php
https://karachitutors.com/teacher-registration.php
https://karachitutors.com/student-registration.php
http://karachitutor

In [11]:
# Find contact forms
forms = soup.find_all("form")

print("Contact/Other Forms Found:", len(forms))

Contact/Other Forms Found: 0


In [12]:
#Finding social media links
social_platforms = [
    "facebook.com",
    "instagram.com",
    "linkedin.com",
    "youtube.com",
    "twitter.com",
    "x.com",
    "tiktok.com"
]

social_links = []

for link in link_urls:
    link_lower = link.lower()
    
    for platform in social_platforms:
        if platform in link_lower:
            social_links.append(link)
            break

print("Social Media Links Found:", len(social_links))

Social Media Links Found: 0


In [13]:
print("Website:", url)
print("Status Code:", response.status_code)
print("Total Links:", len(link_urls))
print("Forms Found:", len(forms))
print("Social Media Links:", len(social_links))

Website: https://karachitutors.com/
Status Code: 200
Total Links: 29
Forms Found: 0
Social Media Links: 0


In [14]:
#Creating the prompt
ai_prompt = """
You are a website quality auditor for Personal Services businesses.

Evaluate the following website based on the factual information provided.

Evaluation criteria:
1. Website Completeness
2. Contact and Business Information
3. Call-to-Action (CTA)
4. Social Media Presence
5. Mobile Responsiveness
6. Visual and Content Quality

Website:
{website}

Automatically detected facts:
- Status Code: {status_code}
- Internal Links Found: {internal_links}
- Contact Information: {contact_info}
- Contact Form: {contact_form}
- Social Media Links: {social_links}
- CTA Indicators: {cta}
- Mobile Responsiveness Indicator: {mobile}

Based only on the available information, provide:

1. Professionalism Score: 0-100
2. Problems: List the main problems found
3. Missing Features: List important missing features
4. Recommendations: Give practical recommendations
5. Priority: Low, Medium, or High

Clearly label the score and recommendations as AI-generated judgment.
Do not present AI judgment as automatically detected facts.
If information is unavailable, clearly mention that it could not be verified.
"""

In [15]:
#Format the prompt with the actual data
ai_output_format = """
Return the evaluation in the following format:

Professionalism Score: [0-100]

Problems:
- [Problem 1]
- [Problem 2]

Missing Features:
- [Missing Feature 1]
- [Missing Feature 2]

Recommendations:
- [Recommendation 1]
- [Recommendation 2]

Priority: [Low/Medium/High]

Label all of the above as AI-generated judgment.
"""

In [16]:
#Combine Prompt Instructions
complete_prompt = ai_prompt + "\n" + ai_output_format

print(complete_prompt)


You are a website quality auditor for Personal Services businesses.

Evaluate the following website based on the factual information provided.

Evaluation criteria:
1. Website Completeness
2. Contact and Business Information
3. Call-to-Action (CTA)
4. Social Media Presence
5. Mobile Responsiveness
6. Visual and Content Quality

Website:
{website}

Automatically detected facts:
- Status Code: {status_code}
- Internal Links Found: {internal_links}
- Contact Information: {contact_info}
- Contact Form: {contact_form}
- Social Media Links: {social_links}
- CTA Indicators: {cta}
- Mobile Responsiveness Indicator: {mobile}

Based only on the available information, provide:

1. Professionalism Score: 0-100
2. Problems: List the main problems found
3. Missing Features: List important missing features
4. Recommendations: Give practical recommendations
5. Priority: Low, Medium, or High

Clearly label the score and recommendations as AI-generated judgment.
Do not present AI judgment as automatica

In [17]:
test_prompt = ai_prompt.format(
    website="https://example.com/",
    status_code=200,
    internal_links=8,
    contact_info="Yes",
    contact_form="No",
    social_links=2,
    cta="Yes",
    mobile="Yes"
)

print(test_prompt)


You are a website quality auditor for Personal Services businesses.

Evaluate the following website based on the factual information provided.

Evaluation criteria:
1. Website Completeness
2. Contact and Business Information
3. Call-to-Action (CTA)
4. Social Media Presence
5. Mobile Responsiveness
6. Visual and Content Quality

Website:
https://example.com/

Automatically detected facts:
- Status Code: 200
- Internal Links Found: 8
- Contact Information: Yes
- Contact Form: No
- Social Media Links: 2
- CTA Indicators: Yes
- Mobile Responsiveness Indicator: Yes

Based only on the available information, provide:

1. Professionalism Score: 0-100
2. Problems: List the main problems found
3. Missing Features: List important missing features
4. Recommendations: Give practical recommendations
5. Priority: Low, Medium, or High

Clearly label the score and recommendations as AI-generated judgment.
Do not present AI judgment as automatically detected facts.
If information is unavailable, clearl

In [18]:
#Final prompt check
required_terms = [
    "Professionalism Score",
    "Problems",
    "Missing Features",
    "Recommendations",
    "Priority"
]

for term in required_terms:
    if term in complete_prompt:
        print(term, "✓")
    else:
        print(term, "✗")

Professionalism Score ✓
Problems ✓
Missing Features ✓
Recommendations ✓
Priority ✓


# Separate Facts from AI-Generated Judgment

The website audit report will clearly separate automatically detected
facts from AI-generated judgment.

Automatically detected facts are collected directly from the website
using Python, Requests, and BeautifulSoup.

AI-generated judgment includes the professionalism score, problems,
missing features, recommendations, and priority.

## 1. Automatically Detected Facts

The following information is collected automatically from the website:

- Website URL
- HTTP status code
- Page title
- Number of internal links
- Number of external links
- Contact information
- Contact form
- Social media links
- CTA indicators
- Mobile responsiveness indicators

These values are treated as factual signals detected by the program.

## 2. AI-Generated Judgment

The AI will separately generate:

- Professionalism Score
- Problems
- Missing Features
- Recommendations
- Priority

These results are AI-generated judgments based on the available
website information and are not treated as automatically detected facts.

In [19]:
#Creating Structure
website_audit = {
    "website": url,
    "automatically_detected_facts": {
        "status_code": response.status_code,
        "total_links": len(link_urls),
        "forms_found": len(forms),
        "social_media_links": len(social_links)
    },
    "ai_generated_judgment": {
        "professionalism_score": None,
        "problems": [],
        "missing_features": [],
        "recommendations": [],
        "priority": None
    }
}

print(website_audit)

{'website': 'https://karachitutors.com/', 'automatically_detected_facts': {'status_code': 200, 'total_links': 29, 'forms_found': 0, 'social_media_links': 0}, 'ai_generated_judgment': {'professionalism_score': None, 'problems': [], 'missing_features': [], 'recommendations': [], 'priority': None}}


# Run Audit on All Selected Websites

The website auditor will now be executed on all selected websites.

For each website, the program will collect automatically detected
factual signals such as status code, links, forms, and social media links.

AI-generated judgment will be added separately after the AI evaluation
stage.

In [20]:
#now we will create a function to automate the collectionof websites data

def audit_website(url):
    try:
        response = requests.get(
            url,
            timeout=10,
            headers={"User-Agent": "Mozilla/5.0"}
        )

        soup = BeautifulSoup(response.text, "html.parser")

        links = soup.find_all("a")

        link_urls = []

        for link in links:
            href = link.get("href")

            if href:
                link_urls.append(href)

        forms = soup.find_all("form")

        social_platforms = [
            "facebook.com",
            "instagram.com",
            "linkedin.com",
            "youtube.com",
            "twitter.com",
            "x.com",
            "tiktok.com"
        ]

        social_links = []

        for link in link_urls:
            link_lower = link.lower()

            for platform in social_platforms:
                if platform in link_lower:
                    social_links.append(link)
                    break

        return {
            "website": url,
            "status_code": response.status_code,
            "total_links": len(link_urls),
            "forms_found": len(forms),
            "social_media_links": len(social_links)
        }

    except Exception as e:
        return {
            "website": url,
            "status_code": "Error",
            "total_links": 0,
            "forms_found": 0,
            "social_media_links": 0,
            "error": str(e)
        }

In [21]:
#Testing the function on a single website
test_result = audit_website(websites[0])

print(test_result)

{'website': 'https://karachitutors.com/', 'status_code': 200, 'total_links': 29, 'forms_found': 0, 'social_media_links': 0}


In [22]:
#now testing all the webs at ones
audit_results = []

for website in websites:
    result = audit_website(website)
    audit_results.append(result)

print("Websites audited:", len(audit_results))

Websites audited: 5


In [23]:
#now every website result in readaable format
for i, result in enumerate(audit_results, start=1):
    print("\nWebsite", i)
    print("-" * 40)

    print("URL:", result["website"])
    print("Status Code:", result["status_code"])
    print("Total Links:", result["total_links"])
    print("Forms Found:", result["forms_found"])
    print("Social Media Links:", result["social_media_links"])


Website 1
----------------------------------------
URL: https://karachitutors.com/
Status Code: 200
Total Links: 29
Forms Found: 0
Social Media Links: 0

Website 2
----------------------------------------
URL: https://dossaniplus.com/
Status Code: 406
Total Links: 0
Forms Found: 0
Social Media Links: 0

Website 3
----------------------------------------
URL: https://fitathome.pk/
Status Code: Error
Total Links: 0
Forms Found: 0
Social Media Links: 0

Website 4
----------------------------------------
URL: https://safaiwala.pk/
Status Code: 200
Total Links: 144
Forms Found: 2
Social Media Links: 6

Website 5
----------------------------------------
URL: https://timesconsultant.com/
Status Code: 200
Total Links: 350
Forms Found: 5
Social Media Links: 6


In [66]:
import json

def generate_ai_judgment(website_data):

    prompt = f"""
You are a professional website quality auditor.

Analyze the website using these automatically detected facts:

{json.dumps(website_data, indent=4)}

Give a balanced evaluation.

Return ONLY valid JSON.
Do NOT use markdown.
Do NOT write ```json.

Use exactly this format:

{{
    "professionalism_score": 0,
    "completeness": "Low",
    "business_functionality": "Low",
    "problems": [],
    "missing_features": [],
    "recommendations": [],
    "priority": "Low"
}}
"""

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    # Check whether Gemini returned anything
    raw_text = response.text

    print("\nRAW AI RESPONSE:")
    print(raw_text)

    if not raw_text or not raw_text.strip():
        return {
            "error": "AI returned an empty response"
        }

    # Remove markdown JSON wrapper if Gemini adds it
    raw_text = raw_text.strip()

    if raw_text.startswith("```json"):
        raw_text = raw_text[7:]

    if raw_text.startswith("```"):
        raw_text = raw_text[3:]

    if raw_text.endswith("```"):
        raw_text = raw_text[:-3]

    raw_text = raw_text.strip()

    try:
        return json.loads(raw_text)

    except json.JSONDecodeError:
        return {
            "error": "AI response was not valid JSON",
            "raw_response": raw_text
        }

In [68]:
website_audit = audit_website("https://safaiwala.pk/")

ai_judgment = generate_ai_judgment(website_audit)




RAW AI RESPONSE:
{
    "professionalism_score": 78,
    "completeness": "High",
    "business_functionality": "Medium",
    "problems": [
        "A high link count relative to basic service sites may dilute navigational focus if not organized cleanly.",
        "Form submission security (such as CAPTCHA or reCAPTCHA) needs validation to prevent automated spam."
    ],
    "missing_features": [
        "Interactive cost estimator or price calculator for cleaning services.",
        "Direct online booking system with calendar date/time selection.",
        "Live chat or direct WhatsApp integration widget for instant customer inquiries."
    ],
    "recommendations": [
        "Integrate a prominent click-to-chat WhatsApp widget to streamline leads in the local market.",
        "Implement direct online scheduling to reduce manual booking handling.",
        "Ensure page speed and mobile responsiveness are optimized across all 144 internal pages and links."
    ],
    "priority": "Mediu

In [47]:
# Convert the flat audit result into the nested report structure
if "automatically_detected_facts" not in website_audit:
	facts = {
		"status_code": website_audit["status_code"],
		"total_links": website_audit["total_links"],
		"forms_found": website_audit["forms_found"],
		"social_media_links": website_audit["social_media_links"]
	}

	website_audit = {
		"website": website_audit["website"],
		"automatically_detected_facts": facts,
		"ai_generated_judgment": ai_judgment
	}

print("WEBSITE AUDIT REPORT")
print("=" * 40)

print("\nAutomatically Detected Facts")
print("-" * 40)

print("Website:", website_audit["website"])
print("Status Code:", website_audit["automatically_detected_facts"]["status_code"])
print("Total Links:", website_audit["automatically_detected_facts"]["total_links"])
print("Forms Found:", website_audit["automatically_detected_facts"]["forms_found"])
print("Social Media Links:", website_audit["automatically_detected_facts"]["social_media_links"])

print("\nAI-Generated Judgment")
print("-" * 40)

print("Professionalism Score:", website_audit["ai_generated_judgment"]["professionalism_score"])
print("Problems:", website_audit["ai_generated_judgment"]["problems"])
print("Missing Features:", website_audit["ai_generated_judgment"]["missing_features"])
print("Recommendations:", website_audit["ai_generated_judgment"]["recommendations"])
print("Priority:", website_audit["ai_generated_judgment"]["priority"])

WEBSITE AUDIT REPORT

Automatically Detected Facts
----------------------------------------
Website: https://safaiwala.pk/
Status Code: 200
Total Links: 144
Forms Found: 2
Social Media Links: 6

AI-Generated Judgment
----------------------------------------
Professionalism Score: 82
Problems: ['High link density (144 links) may cause visual clutter and dilute navigation hierarchy.', 'Limited form integration (only 2 forms) restricts specialized inquiry and conversion paths.']
Missing Features: ['Interactive online booking and scheduling calendar', 'Instant cleaning service price calculator', 'Live chat widget for immediate customer support']
Recommendations: ['Consolidate internal links to streamline user navigation and improve mobile UX.', 'Implement an automated booking system to convert leads directly on the site.', 'Add LocalBusiness structured data (Schema.org) to enhance local search engine rankings.']
Priority: Medium


In [69]:
import json

urls = [
    "https://karachitutors.com/",
    "https://dossaniplus.com/",
    "https://fitathome.pk/",
    "https://safaiwala.pk/",
    "https://timesconsultant.com/"
]

all_reports = []

for url in urls:

    print("\n" + "=" * 80)
    print("AUDITING WEBSITE:", url)
    print("=" * 80)

    try:
        facts = audit_website(url)

        print("\nAUTOMATICALLY DETECTED FACTS")
        print("-" * 50)
        print(json.dumps(facts, indent=4, ensure_ascii=False))

        ai_result = generate_ai_judgment(facts)

        print("\nAI-GENERATED JUDGMENT")
        print("-" * 50)
        print(json.dumps(ai_result, indent=4, ensure_ascii=False))

        all_reports.append({
            "website": url,
            "automatically_detected_facts": facts,
            "ai_generated_judgment": ai_result
        })

    except Exception as e:
        print("\nERROR:", e)

print("\n" + "=" * 80)
print("TOTAL WEBSITES CHECKED:", len(all_reports))
print("=" * 80)


AUDITING WEBSITE: https://karachitutors.com/

AUTOMATICALLY DETECTED FACTS
--------------------------------------------------
{
    "website": "https://karachitutors.com/",
    "status_code": 200,
    "total_links": 29,
    "forms_found": 0,
    "social_media_links": 0
}

ERROR: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 24.742694019s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.c

In [63]:
#Manually reviewing each AI-generated score against my own judgment
manual_review = [
    {
        "website": "https://karachitutors.com/",
        "ai_score": 80,
        "my_score": 75,
        "reason": "I would reduce the score because the website is functional, but some information and call-to-action elements could be presented more clearly."
    },
    {
        "website": "https://safaiwala.pk/",
        "ai_score": 72,
        "my_score": 78,
        "reason": "I would increase the score because the website provides clear service information and useful business functionality."
    }
]

print("=" * 70)
print("MANUAL REVIEW OF AI-GENERATED SCORES")
print("=" * 70)

for review in manual_review:
    adjustment = review["my_score"] - review["ai_score"]

    print("\nWebsite:", review["website"])
    print("AI Score:", review["ai_score"])
    print("My Score:", review["my_score"])
    print("Adjustment:", adjustment)
    print("Why:", review["reason"])

MANUAL REVIEW OF AI-GENERATED SCORES

Website: https://karachitutors.com/
AI Score: 80
My Score: 75
Adjustment: -5
Why: I would reduce the score because the website is functional, but some information and call-to-action elements could be presented more clearly.

Website: https://safaiwala.pk/
AI Score: 72
My Score: 78
Adjustment: 6
Why: I would increase the score because the website provides clear service information and useful business functionality.


In [70]:
#Improve your prompt or logic based on what you found, and re-run to confirm it's more accurate.
import json

def generate_ai_judgment(website_data):

    prompt = f"""
You are an expert website quality auditor.

Evaluate the website using the automatically detected facts below.

IMPORTANT RULES:

1. Give a professionalism score from 0 to 100.
2. Give a completeness assessment.
3. Give a business functionality assessment.
4. Do not give a high score only because the website is accessible.
5. Consider the number of links, forms, social media links, contact/business functionality,
   and overall available evidence.
6. Do not assume that a feature exists if it was not detected.
7. Clearly mention missing features.
8. Recommendations must be practical and based on detected problems.
9. The score should be balanced and not unnecessarily high.
10. Return ONLY valid JSON.

Automatically detected facts:

{json.dumps(website_data, indent=4)}

Return exactly this structure:

{{
    "professionalism_score": 0,
    "completeness": "Low/Medium/High",
    "business_functionality": "Low/Medium/High",
    "problems": [],
    "missing_features": [],
    "recommendations": [],
    "priority": "Low/Medium/High"
}}
"""

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    return json.loads(response.text)

In [83]:
import json

test_url = "https://karachitutors.com/"

try:
    print("1. Checking website...")
    facts = audit_website(test_url)

    print("\n2. Automatically Detected Facts:")
    print(json.dumps(facts, indent=4, ensure_ascii=False))

    print("\n3. Calling Gemini AI...")
    improved_result = generate_ai_judgment(facts)

    print("\n4. IMPROVED AI JUDGMENT")
    print("=" * 50)
    print(json.dumps(improved_result, indent=4, ensure_ascii=False))

except Exception as e:
    print("\nERROR:")
    print(type(e).__name__)
    print(str(e))

1. Checking website...

2. Automatically Detected Facts:
{
    "website": "https://karachitutors.com/",
    "status_code": 200,
    "total_links": 29,
    "forms_found": 0,
    "social_media_links": 0
}

3. Calling Gemini AI...

ERROR:
ClientError
400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}


In [ ]:
#Old vs Improved
old_score = 80
new_score = improved_result["professionalism_score"]

print("Old AI Score:", old_score)
print("Improved AI Score:", new_score)
print("Difference:", new_score - old_score)

In [ ]:
print("\nIMPROVEMENT CONFIRMATION")
print("=" * 50)

if new_score != old_score:
    print("The prompt was improved based on the manual review.")
    print("The AI generated a revised score after considering the detected facts more carefully.")
else:
    print("The revised prompt produced the same score.")
    print("The result was reviewed and considered consistent with the previous assessment.")